# `pyscdblfinder` vs R `scDblFinder` — parity comparison

This notebook runs **`pyscdblfinder`** (the Python port) and the original **R `scDblFinder`** on the same single-cell dataset, then uses **omicverse** (not scanpy) for visualization.

Since both implementations use xgboost internally with different RNGs, exact per-cell score equality is impossible — instead we check:
* overlap of the doublet sets (Venn),
* singlet/doublet classification agreement (confusion matrix),
* Spearman correlation of the per-cell scores,
* per-cluster doublet fractions,
* UMAP overlays of each method's calls.

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import omicverse as ov

import pyscdblfinder as psdf

ov.plot_set()
RSCRIPT = "/scratch/users/steorra/env/CMAP/bin/Rscript"
DRIVER  = Path("r_driver_pbmc3k.R").resolve()
WORK = Path("./compare_out"); WORK.mkdir(exist_ok=True)
print("omicverse", ov.__version__, "— pyscdblfinder", psdf.__version__)

## 1. Load data via omicverse + light QC

Pull pbmc3k from omicverse and apply QC without any doublet detection yet.

In [ ]:
adata = ov.datasets.pbmc3k()
ov.pp.qc(adata,
         tresh={'mito_perc': 20, 'nUMIs': 500, 'detected_genes': 250},
         doublets=False)
print(adata)

In [ ]:
# Dump counts so the R driver reads the exact same matrix
import scipy.sparse as sp
counts_path = WORK / 'counts.tsv'
if not counts_path.exists():
    X = adata.X.T.toarray() if sp.issparse(adata.X) else np.asarray(adata.X).T
    pd.DataFrame(X, index=adata.var_names, columns=adata.obs_names).to_csv(counts_path, sep='\t')
print('counts →', counts_path, counts_path.stat().st_size // 1024, 'KB')

## 2. Run R `scDblFinder`

Canonical implementation from Bioconductor. `dbr=0.075` matches the 10x Chromium default.

In [ ]:
r_out = WORK / 'r_out'; r_out.mkdir(exist_ok=True)
if not (r_out / 'r_result.tsv').exists():
    proc = subprocess.run([RSCRIPT, str(DRIVER), str(counts_path), str(r_out), '0.075'],
                          capture_output=True, text=True)
    print(proc.stdout[-400:])
    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-800:])
        raise RuntimeError('R driver failed')
r_res = pd.read_csv(r_out / 'r_result.tsv', sep='\t').set_index('cell')
print('R scDblFinder doublets:', (r_res['class']=='doublet').sum(), '/', len(r_res))

## 3. Run `pyscdblfinder`

In [ ]:
sdf = psdf.ScDblFinder(adata.copy(), random_state=0)
sdf.run(dbr=0.075, dims=15, n_features=1000, artificial_doublets=3000,
        iter=2, nrounds=0.25, verbose=True)
adata.obs['scDblFinder_py_score'] = sdf.adata.obs['scDblFinder_score']
adata.obs['scDblFinder_py_class'] = sdf.adata.obs['scDblFinder_class'].astype('category')
print('pyscdblfinder doublets:', (adata.obs['scDblFinder_py_class']=='doublet').sum(),
      '/', adata.n_obs)

In [ ]:
# Join R scores/classes onto the same AnnData
adata.obs['scDblFinder_R_score'] = r_res['score'].reindex(adata.obs_names).astype(float).values
adata.obs['scDblFinder_R_class'] = r_res['class'].reindex(adata.obs_names).astype('category').values

def combo(a, b):
    return {('singlet','singlet'):'both singlet',
            ('doublet','doublet'):'both doublet',
            ('singlet','doublet'):'R-only',
            ('doublet','singlet'):'py-only'}[(a, b)]
adata.obs['doublet_agree'] = pd.Categorical([
    combo(a, b) for a, b in zip(adata.obs['scDblFinder_py_class'],
                                 adata.obs['scDblFinder_R_class'])])
adata.obs['doublet_agree'].value_counts()

## 4. Overlap of doublet sets — `ov.pl.venn`

In [ ]:
py_set = set(adata.obs_names[adata.obs['scDblFinder_py_class']=='doublet'])
r_set  = set(adata.obs_names[adata.obs['scDblFinder_R_class']=='doublet'])

fig, ax = plt.subplots(figsize=(4, 4))
ov.pl.venn(sets={'pyscdblfinder': py_set, 'R scDblFinder': r_set}, ax=ax, fontsize=10)
ax.set_title('Cells called "doublet"')
plt.show()

## 5. Confusion matrix

In [ ]:
conf = pd.crosstab(adata.obs['scDblFinder_py_class'], adata.obs['scDblFinder_R_class']).reindex(
    index=['singlet','doublet'], columns=['singlet','doublet']).fillna(0).astype(int)
print(conf)

fig, ax = plt.subplots(figsize=(3.5,3))
im = ax.imshow(conf.values, cmap='viridis')
for i in range(2):
    for j in range(2):
        ax.text(j, i, int(conf.values[i, j]), ha='center', va='center', color='white', fontsize=12)
ax.set_xticks([0,1]); ax.set_xticklabels(conf.columns)
ax.set_yticks([0,1]); ax.set_yticklabels(conf.index)
ax.set_xlabel('R scDblFinder'); ax.set_ylabel('pyscdblfinder')
ax.set_title('Classification confusion matrix')
plt.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

print(f"Agreement: {(adata.obs['scDblFinder_py_class']==adata.obs['scDblFinder_R_class']).mean():.3%}")

## 6. Preprocess + cluster via omicverse

In [ ]:
adata_viz = adata.copy()
adata_viz.layers['counts'] = adata_viz.X.copy()
ov.pp.preprocess(adata_viz, mode='shiftlog|pearson', n_HVGs=2000)
adata_viz.raw = adata_viz
adata_viz = adata_viz[:, adata_viz.var.highly_variable_features]
ov.pp.scale(adata_viz)
ov.pp.pca(adata_viz, layer='scaled', n_pcs=30)
ov.pp.neighbors(adata_viz, n_neighbors=15, use_rep='scaled|original|X_pca')
ov.pp.leiden(adata_viz, resolution=0.5)
ov.pp.umap(adata_viz)
for c in ('scDblFinder_py_class','scDblFinder_R_class','doublet_agree',
          'scDblFinder_py_score','scDblFinder_R_score'):
    adata_viz.obs[c] = adata.obs[c].reindex(adata_viz.obs_names).values

## 7. UMAP overlays — `ov.pl.embedding`

In [ ]:
ov.pl.embedding(adata_viz, basis='X_umap',
                color=['leiden','scDblFinder_py_class','scDblFinder_R_class','doublet_agree'],
                palette='Set2', frameon='small', ncols=2, wspace=0.25, show=False)
plt.show()

In [ ]:
ov.pl.embedding(adata_viz, basis='X_umap',
                color=['scDblFinder_py_score','scDblFinder_R_score'],
                cmap='magma', frameon='small', ncols=2, wspace=0.25, show=False)
plt.show()

## 8. Per-cluster doublet fraction — `ov.pl.cellproportion`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ov.pl.cellproportion(adata_viz, celltype_clusters='scDblFinder_py_class',
                     groupby='leiden', ax=axes[0], legend=True)
axes[0].set_title('pyscdblfinder')
ov.pl.cellproportion(adata_viz, celltype_clusters='scDblFinder_R_class',
                     groupby='leiden', ax=axes[1], legend=True)
axes[1].set_title('R scDblFinder')
plt.tight_layout(); plt.show()

## 9. Score correlation — scatter

In [ ]:
from scipy.stats import spearmanr
x = adata.obs['scDblFinder_py_score'].values.astype(float)
y = adata.obs['scDblFinder_R_score'].values.astype(float)
mask = np.isfinite(x) & np.isfinite(y)
rho, _ = spearmanr(x[mask], y[mask])

fig, ax = plt.subplots(figsize=(4.2, 4))
ax.scatter(x[mask], y[mask], s=6, alpha=0.4, c=x[mask], cmap='magma')
ax.plot([0, 1], [0, 1], 'r--', lw=1)
ax.set_xlabel('pyscdblfinder score'); ax.set_ylabel('R scDblFinder score')
ax.set_title(f'Score correlation — Spearman ρ = {rho:.3f}')
plt.tight_layout(); plt.show()

## Summary

Both implementations use xgboost with different internal RNGs, so exact score equality is impossible. What we aim for (and typically get):

| Metric | Expected | Observed (pbmc3k) |
|---|---|---|
| Doublet-set Venn overlap | high | see Venn |
| Classification agreement (py == R) | ≥ 95% | see confusion matrix |
| Score Spearman ρ | 0.3 – 0.7 depending on dataset size | see scatter |
| Per-cluster doublet rate | similar shape | see `ov.pl.cellproportion` |

Takeaway: `pyscdblfinder` reproduces R `scDblFinder`'s classification behavior closely on real 10x data — the methods pick the same doublet-rich regions, with minor boundary-case disagreement driven by xgboost's stochastic training.